# Notebook 02 — Data Cleaning & Feature Engineering

This notebook cleans all 4 raw datasets and merges them into a single master modelling dataset.

**Input (data/raw/):**
- `kenya_weather_all_counties.csv` — 409,811 rows, NASA POWER daily weather
- `kenya_ipc.csv` — 640 rows, FEWS NET food security classifications
- `knbs_cpi_raw_text.csv` — 37 rows, raw KNBS PDF text
- `kenya_agri_news_raw.csv` — 300 rows, scraped news headlines

**Output (data/processed/):**
- `nasa_monthly_clean.csv` — monthly weather aggregates per county
- `ipc_clean.csv` — clean county-level IPC phases
- `knbs_cpi_structured.csv` — structured price table extracted from PDF text
- `news_clean.csv` — cleaned news articles with NLP features
- `master_dataset.csv` — all datasets merged on county + year + month

In [60]:
import os

# Set working directory to project root using absolute path
project_root = r"C:\Users\HomePC\Desktop\kenya-smart-agriculture"
os.chdir(project_root)

print("Working directory:", os.getcwd())
print("NASA file exists:", os.path.exists("data/raw/weather/kenya_weather_all_counties.csv"))
print("IPC file exists:", os.path.exists("data/raw/food_security/kenya_ipc.csv"))
print("KNBS file exists:", os.path.exists("data/raw/prices/knbs_cpi_raw_text.csv"))
print("News file exists:", os.path.exists("data/raw/news/kenya_agri_news_raw.csv"))

Working directory: C:\Users\HomePC\Desktop\kenya-smart-agriculture
NASA file exists: True
IPC file exists: True
KNBS file exists: True
News file exists: True


In [61]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────
RAW       = "data/raw"
PROCESSED = "data/processed"
os.makedirs(PROCESSED, exist_ok=True)

print("Libraries loaded ✅")
print(f"Output folder: {PROCESSED}/")

Libraries loaded ✅
Output folder: data/processed/


---
## 1. NASA POWER — Weather Data Cleaning

**Goal:** Clean daily weather → aggregate to monthly county-level summaries with drought features.

**Steps:**
1. Parse date strings → datetime
2. Add month + season columns (MAM=long rains, OND=short rains)
3. Check for -999 fill values → replace with NaN
4. Aggregate daily → monthly (sum rainfall, mean temperature)
5. Compute SPI-3 drought index per county
6. Engineer extra features (temp range, dry days, drought flag)


In [62]:
# ── Load raw NASA data ─────────────────────────────────────────
nasa = pd.read_csv(f"{RAW}/weather/kenya_weather_all_counties.csv")
print(f"Raw NASA shape: {nasa.shape}")
print(f"Columns: {list(nasa.columns)}")
print(f"\nSample:")
nasa.head(3)


Raw NASA shape: (409811, 12)
Columns: ['county', 'latitude', 'longitude', 'year', 'date', 'T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'ALLSKY_SFC_SW_DWN']

Sample:


,county,latitude,longitude,year,date,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN
0,Mombasa,-4.0435,39.6682,2000,2000-01-01,28.04,35.60,22.32,0.02,68.84,3.08,24.85
1,Mombasa,-4.0435,39.6682,2000,2000-01-02,27.73,33.56,23.93,0.03,71.04,3.20,23.85
2,Mombasa,-4.0435,39.6682,2000,2000-01-03,27.63,33.72,23.86,0.05,69.03,3.40,23.35


In [63]:
# ── Step 1: Parse dates ────────────────────────────────────────
nasa['date']  = pd.to_datetime(nasa['date'], errors='coerce')
nasa['month'] = nasa['date'].dt.month
nasa['day']   = nasa['date'].dt.day

# Confirm year column already exists
print(f"Date range: {nasa['date'].min().date()} → {nasa['date'].max().date()}")
print(f"Counties  : {nasa['county'].nunique()}")
print(f"Missing dates: {nasa['date'].isnull().sum()}")

Date range: 2000-01-01 → 2023-12-31
Counties  : 47
Missing dates: 0


In [64]:
# ── Step 2: Check for -999 NASA fill values ───────────────────
weather_cols = ['T2M','T2M_MAX','T2M_MIN','PRECTOTCORR','RH2M','WS2M','ALLSKY_SFC_SW_DWN']

before = (nasa[weather_cols] == -999.0).sum().sum()
nasa[weather_cols] = nasa[weather_cols].replace(-999.0, np.nan)
after  = nasa[weather_cols].isnull().sum().sum()

print(f"-999 fill values replaced with NaN: {before} found")
print(f"\nMissing data after replacement (%):")
print(nasa[weather_cols].isnull().mean().mul(100).round(2).to_string())

-999 fill values replaced with NaN: 0 found

Missing data after replacement (%):
T2M                  0.0
T2M_MAX              0.0
T2M_MIN              0.0
PRECTOTCORR          0.0
RH2M                 0.0
WS2M                 0.0
ALLSKY_SFC_SW_DWN    0.0


In [65]:
# ── Step 3: Add season column ──────────────────────────────────
# Kenya has 2 main crop seasons:
#   MAM = March-April-May (Long rains) — main planting season
#   OND = October-November-December (Short rains) — second planting
#   DS1 = January-February (Dry season)
#   DS2 = June-September (Dry season)

season_map = {1:'DS1', 2:'DS1', 3:'MAM', 4:'MAM', 5:'MAM',
              6:'DS2', 7:'DS2', 8:'DS2', 9:'DS2',
              10:'OND', 11:'OND', 12:'OND'}

nasa['season'] = nasa['month'].map(season_map)

print("Season distribution:")
print(nasa['season'].value_counts().sort_index().to_string())


Season distribution:
season
DS1     66479
DS2    136884
MAM    103224
OND    103224


In [66]:
# ── Step 4: Aggregate daily → monthly ─────────────────────────
nasa_monthly = nasa.groupby(['county','year','month','season']).agg(
    total_rainfall    = ('PRECTOTCORR',      'sum'),
    mean_temp         = ('T2M',              'mean'),
    max_temp          = ('T2M_MAX',          'mean'),
    min_temp          = ('T2M_MIN',          'mean'),
    mean_humidity     = ('RH2M',             'mean'),
    mean_solar        = ('ALLSKY_SFC_SW_DWN','mean'),
    mean_wind         = ('WS2M',             'mean'),
    dry_days          = ('PRECTOTCORR', lambda x: (x < 1.0).sum()),
    obs_count         = ('date',             'count'),
).reset_index()

print(f"Monthly shape: {nasa_monthly.shape}")
print(f"Sample:")
nasa_monthly.head(3)


Monthly shape: (13464, 13)
Sample:


,county,year,month,season,total_rainfall,mean_temp,max_temp,min_temp,mean_humidity,mean_solar,mean_wind,dry_days,obs_count
0,Baringo,2000,1,DS1,4.35,21.888065,30.852258,14.214194,47.369677,25.463871,3.561290,30,31
1,Baringo,2000,2,DS1,2.19,23.051034,32.348966,14.752759,38.531379,27.362759,3.456552,28,29
2,Baringo,2000,3,MAM,5.66,24.026774,32.907097,16.085806,45.580645,25.525484,3.637742,30,31


In [67]:
# ── Step 5: Feature engineering ───────────────────────────────

# Temperature range (daily variability — stress indicator for crops)
nasa_monthly['temp_range'] = nasa_monthly['max_temp'] - nasa_monthly['min_temp']

# Drought flag (binary: 1 if more than half the month was dry)
nasa_monthly['drought_flag'] = (
    nasa_monthly['dry_days'] > 15
).astype(int)

# Season flags (useful as ML binary features)
nasa_monthly['is_long_rains']  = (nasa_monthly['season'] == 'MAM').astype(int)
nasa_monthly['is_short_rains'] = (nasa_monthly['season'] == 'OND').astype(int)

print("New features added: temp_range, drought_flag, is_long_rains, is_short_rains")

New features added: temp_range, drought_flag, is_long_rains, is_short_rains


In [68]:
# ── Step 6: Compute SPI-3 (Standardised Precipitation Index) ──
# SPI-3 = (rainfall - 3-month rolling mean) / 3-month rolling std
# Positive = wetter than normal, Negative = drier (drought)
# Values below -1.0 = drought warning threshold

nasa_monthly = nasa_monthly.sort_values(['county','year','month'])

def compute_spi(series, window=3):
    rolling_mean = series.rolling(window=window, min_periods=1).mean()
    rolling_std  = series.rolling(window=window, min_periods=1).std()
    return (series - rolling_mean) / rolling_std.replace(0, np.nan)

nasa_monthly['spi_3'] = (
    nasa_monthly.groupby('county')['total_rainfall']
    .transform(lambda x: compute_spi(x, window=3))
)

print("SPI-3 computed ✅")
print(f"\nSPI-3 distribution:")
print(nasa_monthly['spi_3'].describe().round(3).to_string())
print(f"\nDrought events (SPI-3 < -1.0): {(nasa_monthly['spi_3'] < -1.0).sum():,}")

SPI-3 computed ✅

SPI-3 distribution:
count    13417.000
mean         0.019
std          0.858
min         -1.155
25%         -0.776
50%         -0.154
75%          0.997
max          1.155

Drought events (SPI-3 < -1.0): 1,904


In [69]:
# ── Save cleaned NASA data ─────────────────────────────────────
nasa_monthly.to_csv(f"{PROCESSED}/nasa_monthly_clean.csv", index=False)

print(f"✅ Saved: {PROCESSED}/nasa_monthly_clean.csv")
print(f"   Shape : {nasa_monthly.shape}")
print(f"   Counties: {nasa_monthly['county'].nunique()}")
print(f"   Date range: {nasa_monthly['year'].min()} – {nasa_monthly['year'].max()}")
print(f"\nColumn list:")
for col in nasa_monthly.columns:
    print(f"  {col:<25} {nasa_monthly[col].dtype}")

✅ Saved: data/processed/nasa_monthly_clean.csv
   Shape : (13464, 18)
   Counties: 47
   Date range: 2000 – 2023

Column list:
  county                    object
  year                      int64
  month                     int32
  season                    object
  total_rainfall            float64
  mean_temp                 float64
  max_temp                  float64
  min_temp                  float64
  mean_humidity             float64
  mean_solar                float64
  mean_wind                 float64
  dry_days                  int64
  obs_count                 int64
  temp_range                float64
  drought_flag              int32
  is_long_rains             int32
  is_short_rains            int32
  spi_3                     float64


---
## 2. FEWS NET IPC — Food Security Cleaning

**Goal:** Clean IPC classification data → one row per county with IPC phase label.

**Steps:**
1. Rename cryptic columns to readable names
2. Parse coverage dates from MM-YYYY format
3. Drop ADMIN3 (55.5% missing — structural gap)
4. Standardise county names to match NASA
5. Aggregate sub-county rows → one IPC phase per county (mode)
6. Add human-readable phase labels


In [70]:
# ── Load raw IPC data ──────────────────────────────────────────
ipc = pd.read_csv(f"{RAW}/food_security/kenya_ipc.csv")
print(f"Raw IPC shape: {ipc.shape}")
print(f"Columns: {list(ipc.columns)}")
print(f"\nSample:")
ipc.head(3)

Raw IPC shape: (640, 16)
Columns: ['cov_start', 'cov_end', 'report_mon', 'country', 'fnid', 'unit_name', 'ADMIN0', 'ADMIN1', 'ADMIN2', 'ADMIN3', 'LZCODE', 'LZNAME', 'ML1', 'HA1', 'unit_type', 'fewsnet_re']

Sample:


,cov_start,cov_end,report_mon,country,fnid,unit_name,ADMIN0,ADMIN1,ADMIN2,ADMIN3,LZCODE,LZNAME,ML1,HA1,unit_type,fewsnet_re
0,03-2026,05-2026,03-2026,KE,KE2016C312010019,"Central Highlands, High Potential Zone, Buuri,...",Kenya,Meru,Buuri,NaN,KE19,"Central Highlands, High Potential Zone",3,False,fsc_admin_lhz,East Africa
1,03-2026,05-2026,03-2026,KE,KE2016C312020019,"Central Highlands, High Potential Zone, Centra...",Kenya,Meru,Central Imenti,NaN,KE19,"Central Highlands, High Potential Zone",1,False,fsc_admin_lhz,East Africa
2,03-2026,05-2026,03-2026,KE,KE2016C313010019,"Central Highlands, High Potential Zone, Chuka/...",Kenya,Tharaka Nithi,Chuka/Igambang'Ombe,NaN,KE19,"Central Highlands, High Potential Zone",1,False,fsc_admin_lhz,East Africa


In [71]:
# ── Step 1: Rename columns ─────────────────────────────────────
col_map = {
    'cov_start':  'period_start',
    'cov_end':    'period_end',
    'report_mon': 'report_month',
    'ADMIN0':     'country',
    'ADMIN1':     'county',
    'ADMIN2':     'sub_county',
    'ADMIN3':     'location',
    'LZCODE':     'livelihood_zone_code',
    'LZNAME':     'livelihood_zone_name',
    'ML1':        'ipc_phase',
    'HA1':        'humanitarian_assistance',
    'unit_type':  'unit_type',
    'fewsnet_re': 'fewsnet_region',
}
ipc = ipc.rename(columns=col_map)
print("Columns renamed ✅")
print(list(ipc.columns))

Columns renamed ✅
['period_start', 'period_end', 'report_month', 'country', 'fnid', 'unit_name', 'country', 'county', 'sub_county', 'location', 'livelihood_zone_code', 'livelihood_zone_name', 'ipc_phase', 'humanitarian_assistance', 'unit_type', 'fewsnet_region']


In [72]:
# ── Step 2: Parse dates ────────────────────────────────────────
# Format is MM-YYYY e.g. "03-2026"
for col in ['period_start','period_end','report_month']:
    ipc[col] = pd.to_datetime(ipc[col], format='%m-%Y', errors='coerce')

print(f"Coverage: {ipc['period_start'].min().date()} → {ipc['period_end'].max().date()}")

Coverage: 2026-03-01 → 2026-05-01


In [73]:
# ── Step 3: Drop ADMIN3 — 55.5% missing (structural gap) ──────
missing_pct = ipc['location'].isnull().mean() * 100
print(f"'location' (ADMIN3) missing: {missing_pct:.1f}% → dropping")
ipc = ipc.drop(columns=['location'])


'location' (ADMIN3) missing: 55.5% → dropping


In [74]:
# ── Step 4: Standardise county names ──────────────────────────
# Map to match NASA county names exactly
county_corrections = {
    "Elgeyo-Marakwet": "Elgeyo Marakwet",
    "Murang'a":        "Muranga",
    "Muranga":         "Muranga",
    "Tharaka-Nithi":   "Tharaka Nithi",
    "Trans-Nzoia":     "Trans Nzoia",
    "West-Pokot":      "West Pokot",
    "Taita-Taveta":    "Taita Taveta",
    "Homa-Bay":        "Homa Bay",
    "Tana-River":      "Tana River",
    "Uasin-Gishu":     "Uasin Gishu",
}
ipc['county'] = ipc['county'].replace(county_corrections)

print(f"Counties in IPC: {sorted(ipc['county'].unique())}")


Counties in IPC: ['Baringo', 'Bomet', 'Bungoma', 'Busia', 'Elgeyo Marakwet', 'Embu', 'Garissa', 'Homa Bay', 'Isiolo', 'Kajiado', 'Kakamega', 'Kericho', 'Kiambu', 'Kilifi', 'Kirinyaga', 'Kisii', 'Kisumu', 'Kitui', 'Kwale', 'Laikipia', 'Lamu', 'Machakos', 'Makueni', 'Mandera', 'Marsabit', 'Meru', 'Migori', 'Mombasa', 'Muranga', 'Nairobi', 'Nakuru', 'Nandi', 'Narok', 'Nyamira', 'Nyandarua', 'Nyeri', 'Samburu', 'Siaya', 'Taita Taveta', 'Tana River', 'Tharaka Nithi', 'Trans Nzoia', 'Turkana', 'Uasin Gishu', 'Vihiga', 'Wajir', 'West Pokot']


In [75]:
# ── Step 5: Add IPC phase labels ───────────────────────────────
phase_labels = {1:'Minimal', 2:'Stressed', 3:'Crisis', 4:'Emergency', 5:'Famine'}
ipc['ipc_phase_label'] = ipc['ipc_phase'].map(phase_labels)

print("IPC phase distribution (all sub-county rows):")
print(ipc['ipc_phase'].value_counts().sort_index().to_string())
print()
print(ipc[['county','sub_county','ipc_phase','ipc_phase_label']].head(10).to_string())


IPC phase distribution (all sub-county rows):
ipc_phase
1    245
2     81
3    314

          county           sub_county  ipc_phase ipc_phase_label
0           Meru                Buuri          3          Crisis
1           Meru       Central Imenti          1         Minimal
2  Tharaka Nithi  Chuka/Igambang'Ombe          1         Minimal
3        Nairobi      Dagoretti North          1         Minimal
4        Nairobi      Dagoretti South          1         Minimal
5        Nairobi     Embakasi Central          1         Minimal
6        Nairobi        Embakasi East          1         Minimal
7        Nairobi       Embakasi North          1         Minimal
8        Nairobi       Embakasi South          1         Minimal
9        Nairobi        Embakasi West          1         Minimal


In [76]:
# ── Step 6: Aggregate → one row per county ────────────────────
# Use mode (most common phase) per county
ipc_county = (
    ipc.groupby('county')
    .agg(
        ipc_phase        = ('ipc_phase', lambda x: int(x.mode()[0])),
        zone_count       = ('livelihood_zone_name', 'count'),
        has_humanitarian = ('humanitarian_assistance', 'any'),
        period_start     = ('period_start', 'first'),
        period_end       = ('period_end', 'first'),
    )
    .reset_index()
)
ipc_county['ipc_phase_label'] = ipc_county['ipc_phase'].map(phase_labels)

print(f"County-level IPC shape: {ipc_county.shape}")
print()
print("IPC phase by county:")
print(ipc_county[['county','ipc_phase','ipc_phase_label']].sort_values('ipc_phase', ascending=False).to_string(index=False))

County-level IPC shape: (47, 7)

IPC phase by county:
         county  ipc_phase ipc_phase_label
        Mandera          3          Crisis
          Kitui          3          Crisis
          Wajir          3          Crisis
        Turkana          3          Crisis
     Tana River          3          Crisis
        Garissa          3          Crisis
        Samburu          3          Crisis
         Isiolo          3          Crisis
           Meru          3          Crisis
       Marsabit          3          Crisis
        Makueni          3          Crisis
           Lamu          3          Crisis
   Taita Taveta          2        Stressed
          Narok          2        Stressed
       Laikipia          2        Stressed
          Kwale          2        Stressed
        Baringo          2        Stressed
        Kajiado          2        Stressed
         Kilifi          2        Stressed
           Embu          2        Stressed
     West Pokot          2        Stressed


In [77]:
# ── Save cleaned IPC data ──────────────────────────────────────
ipc.to_csv(f"{PROCESSED}/ipc_clean.csv", index=False)
ipc_county.to_csv(f"{PROCESSED}/ipc_county.csv", index=False)

print(f"✅ Saved: {PROCESSED}/ipc_clean.csv       ({len(ipc):,} rows — sub-county level)")
print(f"✅ Saved: {PROCESSED}/ipc_county.csv      ({len(ipc_county):,} rows — county level)")

✅ Saved: data/processed/ipc_clean.csv       (640 rows — sub-county level)
✅ Saved: data/processed/ipc_county.csv      (47 rows — county level)


---
## 3. KNBS CPI — Extract Prices from Raw PDF Text

**This is the most technically complex step.**

The raw data is 37 PDF reports converted to text. Each report's Table 1 contains
a historical CPI time series. We use regex to extract:
1. Month + Overall CPI + Inflation Rate (from Table 1)
2. Key commodity prices: Sugar, Maize, Tomatoes (from commodity tables)

Each report contains ~12–15 months of CPI history, giving us a rich time series.

In [78]:
# ── Load raw KNBS text ─────────────────────────────────────────
knbs_raw = pd.read_csv(f"{RAW}/prices/knbs_cpi_raw_text.csv")
print(f"Raw KNBS shape: {knbs_raw.shape}")
print(f"\nFiles available:")
for f in knbs_raw['file'].tolist():
    print(f"  {f}")

Raw KNBS shape: (37, 2)

Files available:
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-April-2021.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-August-2024.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-December-2021.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-December-2022.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-December-2025.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-Highlights-April-2024.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-January-2021.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-July-2020.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-July-2023.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-June-2020.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-June-2021.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-June-2023.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-June-2025.pdf
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-March-2020.pdf
  Kenya-Consumer-

In [79]:
# ── Regex patterns for KNBS CPI extraction ────────────────────

# Month name mapping
MONTH_MAP = {
    'january':1,'february':2,'march':3,'april':4,'may':5,'june':6,
    'july':7,'august':8,'september':9,'october':10,'november':11,'december':12
}

def parse_filename(filename):
    """Extract month and year from PDF filename."""
    fn = filename.lower()
    year_match = re.search(r'(20\d{2})', fn)
    year = int(year_match.group(1)) if year_match else None
    month = next((v for k,v in MONTH_MAP.items() if k in fn), None)
    return month, year

# Test filename parsing
for fn in knbs_raw['file'].head(5):
    m, y = parse_filename(fn)
    print(f"  {fn[:60]:<60} → {m}/{y}")


  Kenya-Consumer-Price-Indices-and-Inflation-Rates-April-2021. → 4/2021
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-August-2024 → 8/2024
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-December-20 → 12/2021
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-December-20 → 12/2022
  Kenya-Consumer-Price-Indices-and-Inflation-Rates-December-20 → 12/2025


In [80]:
# ── Extract CPI time series from Table 1 ─────────────────────
# Pattern: "Month Name YYYY  CPI_VALUE  INFLATION_RATE"
# Example: "April 2021  114.746  5.76"

CPI_TABLE_PATTERN = re.compile(
    r'(January|February|March|April|May|June|July|August|'
    r'September|October|November|December)\s+(20\d{2})\s+'
    r'([\d,]+\.?\d*)\s+([\d.]+)',
    re.IGNORECASE
)

all_rows = []

for _, row in knbs_raw.iterrows():
    content  = str(row['content'])
    src_file = row['file']

    matches = CPI_TABLE_PATTERN.findall(content)

    for month_name, year_str, cpi_str, inf_str in matches:
        month = MONTH_MAP.get(month_name.lower())
        year  = int(year_str)
        try:
            cpi       = float(cpi_str.replace(',',''))
            inflation = float(inf_str)
            all_rows.append({
                'month':       month,
                'year':        year,
                'month_name':  month_name.title(),
                'overall_cpi': cpi,
                'yoy_inflation': inflation,
                'source_file': src_file,
            })
        except ValueError:
            pass

cpi_df = pd.DataFrame(all_rows)

# Deduplicate — same month/year may appear in multiple reports
cpi_df = (cpi_df
    .sort_values('source_file')
    .drop_duplicates(subset=['month','year'], keep='last')
    .sort_values(['year','month'])
    .reset_index(drop=True)
)

cpi_df['date'] = pd.to_datetime(
    cpi_df[['year','month']].assign(day=1)
)

print(f"CPI time series extracted: {len(cpi_df)} monthly records")
print(f"Date range: {cpi_df['date'].min().date()} → {cpi_df['date'].max().date()}")
print()
print(cpi_df[['date','overall_cpi','yoy_inflation']].head(10).to_string(index=False))

CPI time series extracted: 65 monthly records
Date range: 2020-01-01 → 2025-05-01

      date  overall_cpi  yoy_inflation
2020-01-01     2020.000        2021.00
2020-02-01      107.174           7.17
2020-03-01      107.475           5.84
2020-04-01      108.495           6.01
2020-05-01      108.602           5.33
2020-06-01      108.266           4.59
2020-07-01      108.354           4.36
2020-08-01      108.573           4.36
2020-09-01      108.571           4.20
2020-10-01      109.604           4.84


In [81]:
# Remove bad rows where CPI value is clearly wrong
# Real Kenya CPI is always between 50 and 500
before = len(cpi_df)
cpi_df = cpi_df[
    (cpi_df['overall_cpi'] > 50) & 
    (cpi_df['overall_cpi'] < 500)
].reset_index(drop=True)

print(f"Removed {before - len(cpi_df)} bad rows")
print(f"Clean records: {len(cpi_df)}")
print(f"Date range: {cpi_df['date'].min().date()} → {cpi_df['date'].max().date()}")
print()
print(cpi_df[['date','overall_cpi','yoy_inflation']].head(10).to_string(index=False))

Removed 1 bad rows
Clean records: 64
Date range: 2020-02-01 → 2025-05-01

      date  overall_cpi  yoy_inflation
2020-02-01      107.174           7.17
2020-03-01      107.475           5.84
2020-04-01      108.495           6.01
2020-05-01      108.602           5.33
2020-06-01      108.266           4.59
2020-07-01      108.354           4.36
2020-08-01      108.573           4.36
2020-09-01      108.571           4.20
2020-10-01      109.604           4.84
2020-11-01      110.779           5.33


In [82]:
# Use CPI time series as the structured KNBS dataset
# Commodity prices not available across enough PDFs to be useful
knbs_clean = cpi_df.copy()

# Add the one report that has commodity data as bonus columns
april_2021 = knbs_raw[knbs_raw['file'].str.contains('April-2021')]['content'].values[0]

def extract_single_value(text, pattern):
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        try:
            return float(match.group(1))
        except:
            return None
    return None

# We have one month of commodity data — note it but don't rely on it
sugar_apr21    = extract_single_value(april_2021, r'Sugar\s+1Kilogramme\s+([\d.]+)')
maize_apr21    = extract_single_value(april_2021, r'Maize Grain.*?1Kilogramme\s+([\d.]+)')
tomatoes_apr21 = extract_single_value(april_2021, r'Tomatoes\s+1Kilogramme\s+([\d.]+)')

print("April 2021 commodity prices (only available month):")
print(f"  Sugar (1kg)   : KES {sugar_apr21}")
print(f"  Maize (1kg)   : KES {maize_apr21}")
print(f"  Tomatoes (1kg): KES {tomatoes_apr21}")
print()
print(f"Main dataset: {len(knbs_clean)} monthly CPI records")
print(f"Date range  : {knbs_clean['date'].min().date()} → {knbs_clean['date'].max().date()}")
print(f"Columns     : {list(knbs_clean.columns)}")

April 2021 commodity prices (only available month):
  Sugar (1kg)   : KES 113.98
  Maize (1kg)   : KES 51.2
  Tomatoes (1kg): KES 119.47

Main dataset: 64 monthly CPI records
Date range  : 2020-02-01 → 2025-05-01
Columns     : ['month', 'year', 'month_name', 'overall_cpi', 'yoy_inflation', 'source_file', 'date']


In [83]:
# Save cleaned KNBS data
knbs_clean.to_csv(f"{PROCESSED}/knbs_cpi_structured.csv", index=False)

print(f"✅ Saved: {PROCESSED}/knbs_cpi_structured.csv")
print(f"   Shape      : {knbs_clean.shape}")
print(f"   Date range : {knbs_clean['date'].min().date()} → {knbs_clean['date'].max().date()}")
print(f"   Target variable for forecasting: overall_cpi")
print(f"   Secondary target: yoy_inflation")
print()
print("Sample:")
print(knbs_clean[['date','overall_cpi','yoy_inflation']].tail(10).to_string(index=False))

✅ Saved: data/processed/knbs_cpi_structured.csv
   Shape      : (64, 7)
   Date range : 2020-02-01 → 2025-05-01
   Target variable for forecasting: overall_cpi
   Secondary target: yoy_inflation

Sample:
      date  overall_cpi  yoy_inflation
2024-08-01       139.87            4.4
2024-09-01       140.13            3.6
2024-10-01       140.44            2.7
2024-11-01       140.81            2.8
2024-12-01       141.66            3.0
2025-01-01       142.68            3.3
2025-02-01       143.12            3.5
2025-03-01       143.69            3.6
2025-04-01       144.09            4.1
2025-05-01       144.88            3.8


---
## 4. News Articles — NLP Preprocessing

**Goal:** Clean scraped headlines for NLP analysis.

**Steps:**
1. Parse ISO 8601 dates (strip timezone)
2. Deduplicate by URL
3. Clean title text (strip HTML artifacts)
4. Extract county mentions from headlines
5. Extract commodity mentions from headlines
6. Add text features (length, word count)

In [84]:
# ── Load raw news ──────────────────────────────────────────────
news = pd.read_csv(f"{RAW}/news/kenya_agri_news_raw.csv")
print(f"Raw news shape: {news.shape}")
print(f"Columns: {list(news.columns)}")
print()
news.head(3)

Raw news shape: (300, 5)
Columns: ['source', 'title', 'url', 'date', 'scraped_at']



,source,title,url,date,scraped_at
0,Kenya News Agency,Sugarcane farming cut to focus on food production,https://www.kenyanews.go.ke/sugarcane-farming-...,2026-05-07T09:14:26+00:00,2026-05-07
1,Kenya News Agency,Coffee revival gains ground,https://www.kenyanews.go.ke/coffee-revival-gai...,2026-05-06T14:16:40+00:00,2026-05-07
2,Kenya News Agency,Young Murang’a women opt for economic independ...,https://www.kenyanews.go.ke/young-muranga-wome...,2026-05-06T10:14:25+00:00,2026-05-07


In [85]:
# ── Step 1: Parse dates ────────────────────────────────────────
# Format: "2026-05-07T09:14:26+00:00" — need to strip timezone
news['date'] = pd.to_datetime(news['date'], utc=True, errors='coerce')
news['date'] = news['date'].dt.tz_localize(None)  # Remove timezone
news['year']  = news['date'].dt.year
news['month'] = news['date'].dt.month

print(f"Date range: {news['date'].min().date()} → {news['date'].max().date()}")
print(f"Missing dates: {news['date'].isnull().sum()}")


Date range: 2025-05-09 → 2026-05-07
Missing dates: 0


In [86]:
# ── Step 2: Deduplicate by URL ─────────────────────────────────
before = len(news)
news = news.drop_duplicates(subset=['url']).reset_index(drop=True)
print(f"Duplicates removed: {before - len(news)} → {len(news)} articles remain")

Duplicates removed: 0 → 300 articles remain


In [87]:
# ── Step 3: Clean title text ───────────────────────────────────
news['title_clean'] = (
    news['title']
    .str.replace(r'<[^>]+>', '', regex=True)   # Remove HTML tags
    .str.replace(r'\s+', ' ', regex=True)       # Collapse whitespace
    .str.strip()
)

# Drop titles that are too short (likely scraping artifacts)
news = news[news['title_clean'].str.len() > 15].reset_index(drop=True)
print(f"After short-title filter: {len(news)} articles")
print()
print("Sample clean titles:")
for t in news['title_clean'].head(8):
    print(f"  {t}")


After short-title filter: 300 articles

Sample clean titles:
  Sugarcane farming cut to focus on food production
  Coffee revival gains ground
  Young Murang’a women opt for economic independence through farming
  Rains choke tomato supply in Kisumu
  Kandie launches annual fruition programme to boost farmers’ livelihoods
  Registration delays slow livestock vaccination in Meru County
  Govt outlines reforms in education, health, economy sectors
  Mandera trains 20 lead farmers to strengthen grassroots agricultural support


In [88]:
# ── Step 4: Extract county mentions ───────────────────────────
KENYA_COUNTIES = [
    'mombasa','kwale','kilifi','tana river','lamu','taita taveta',
    'garissa','wajir','mandera','marsabit','isiolo','meru',
    'tharaka nithi','embu','kitui','machakos','makueni','nyandarua',
    'nyeri','kirinyaga','muranga','kiambu','turkana','west pokot',
    'samburu','trans nzoia','uasin gishu','elgeyo marakwet','nandi',
    'baringo','laikipia','nakuru','narok','kajiado','kericho','bomet',
    'kakamega','vihiga','bungoma','busia','siaya','kisumu',
    'homa bay','migori','kisii','nyamira','nairobi',
]

COMMODITIES = [
    'maize','beans','sugar','flour','rice','tomatoes','onions',
    'potatoes','kale','sukuma','milk','cooking oil','wheat',
    'sorghum','tea','coffee','avocado','cassava',
]

def find_mentions(title, keywords):
    title_lower = title.lower()
    return [k.title() for k in keywords if k in title_lower]

news['counties_mentioned']    = news['title_clean'].apply(lambda t: find_mentions(t, KENYA_COUNTIES))
news['commodities_mentioned'] = news['title_clean'].apply(lambda t: find_mentions(t, COMMODITIES))
news['county_count']          = news['counties_mentioned'].apply(len)
news['commodity_count']       = news['commodities_mentioned'].apply(len)

print(f"Articles mentioning a county    : {(news['county_count'] > 0).sum()}")
print(f"Articles mentioning a commodity : {(news['commodity_count'] > 0).sum()}")
print()
print("Top counties mentioned in headlines:")
from collections import Counter
county_mentions = Counter(c for lst in news['counties_mentioned'] for c in lst)
print(pd.Series(county_mentions).sort_values(ascending=False).head(10).to_string())

Articles mentioning a county    : 94
Articles mentioning a commodity : 76

Top counties mentioned in headlines:
Vihiga             12
Turkana             9
Kirinyaga           6
Migori              5
Busia               5
Nakuru              5
Kwale               5
West Pokot          4
Elgeyo Marakwet     4
Kericho             4


In [89]:
# ── Step 5: Add text features ──────────────────────────────────
news['title_length'] = news['title_clean'].str.len()
news['word_count']   = news['title_clean'].str.split().str.len()

print("Text feature summary:")
print(news[['title_length','word_count']].describe().round(1).to_string())

Text feature summary:
       title_length  word_count
count         300.0       300.0
mean           59.2         8.5
std            11.8         1.8
min            27.0         4.0
25%            51.0         7.0
50%            59.0         8.0
75%            67.0         9.0
max            93.0        15.0


In [90]:
# ── Save cleaned news data ─────────────────────────────────────
news.to_csv(f"{PROCESSED}/news_clean.csv", index=False)
print(f"✅ Saved: {PROCESSED}/news_clean.csv")
print(f"   Shape   : {news.shape}")
print(f"   Columns : {list(news.columns)}")


✅ Saved: data/processed/news_clean.csv
   Shape   : (300, 14)
   Columns : ['source', 'title', 'url', 'date', 'scraped_at', 'year', 'month', 'title_clean', 'counties_mentioned', 'commodities_mentioned', 'county_count', 'commodity_count', 'title_length', 'word_count']


---
## 5. Build Master Dataset

**Goal:** Merge all cleaned datasets into one modelling-ready master DataFrame.

**Join strategy:**
- NASA monthly (county + year + month) is the base table
- KNBS CPI joins on (year + month) — national level, broadcast to all counties
- IPC county phase joins on (county) — single snapshot from March 2026

In [91]:
# ── Load all cleaned datasets ──────────────────────────────────
nasa_m   = pd.read_csv(f"{PROCESSED}/nasa_monthly_clean.csv")
ipc_c    = pd.read_csv(f"{PROCESSED}/ipc_county.csv")
knbs_c   = pd.read_csv(f"{PROCESSED}/knbs_cpi_structured.csv")

print(f"NASA monthly : {nasa_m.shape}")
print(f"IPC county   : {ipc_c.shape}")
print(f"KNBS CPI     : {knbs_c.shape}")


NASA monthly : (13464, 18)
IPC county   : (47, 7)
KNBS CPI     : (64, 7)


In [92]:
# ── Merge 1: NASA + KNBS CPI (national price data) ────────────
# CPI is national-level → joins on year + month (broadcasts to all counties)
cpi_cols = ['year','month','overall_cpi','yoy_inflation',
            'sugar_kg','maize_grain_kg','tomatoes_kg']
cpi_cols = [c for c in cpi_cols if c in knbs_c.columns]

master = nasa_m.merge(
    knbs_c[cpi_cols],
    on=['year','month'],
    how='left'
)
print(f"After NASA + KNBS merge: {master.shape}")
print(f"CPI coverage: {master['overall_cpi'].notna().mean()*100:.1f}% of rows have a CPI value")

After NASA + KNBS merge: (13464, 20)
CPI coverage: 16.1% of rows have a CPI value


In [93]:
# ── Merge 2: Add IPC phase per county ─────────────────────────
ipc_merge = ipc_c[['county','ipc_phase','ipc_phase_label','has_humanitarian']].copy()

master = master.merge(ipc_merge, on='county', how='left')
print(f"After IPC merge: {master.shape}")
print(f"IPC coverage: {master['ipc_phase'].notna().mean()*100:.1f}% of rows have an IPC phase")

After IPC merge: (13464, 23)
IPC coverage: 100.0% of rows have an IPC phase


In [94]:
# ── Final master dataset summary ──────────────────────────────
print(f"\n{'='*55}")
print(f"  MASTER DATASET SUMMARY")
print(f"{'='*55}")
print(f"  Shape          : {master.shape}")
print(f"  Counties       : {master['county'].nunique()}")
print(f"  Date range     : {master['year'].min()} – {master['year'].max()}")
print(f"  Total rows     : {len(master):,}")
print(f"\n  Column list:")
for col in master.columns:
    missing = master[col].isnull().mean() * 100
    flag = " ⚠️" if missing > 50 else ""
    print(f"    {col:<30} {master[col].dtype}   {missing:.0f}% missing{flag}")



  MASTER DATASET SUMMARY
  Shape          : (13464, 23)
  Counties       : 47
  Date range     : 2000 – 2023
  Total rows     : 13,464

  Column list:
    county                         object   0% missing
    year                           int64   0% missing
    month                          int64   0% missing
    season                         object   0% missing
    total_rainfall                 float64   0% missing
    mean_temp                      float64   0% missing
    max_temp                       float64   0% missing
    min_temp                       float64   0% missing
    mean_humidity                  float64   0% missing
    mean_solar                     float64   0% missing
    mean_wind                      float64   0% missing
    dry_days                       int64   0% missing
    obs_count                      int64   0% missing
    temp_range                     float64   0% missing
    drought_flag                   int64   0% missing
    is_long_rains   

In [95]:
# ── Save master dataset ────────────────────────────────────────
master.to_csv(f"{PROCESSED}/master_dataset.csv", index=False)
print(f"\n✅ Saved: {PROCESSED}/master_dataset.csv")
print(f"\n{'='*55}")
print(f"  ALL PROCESSED FILES")
print(f"{'='*55}")
for f in os.listdir(PROCESSED):
    if f.endswith('.csv'):
        path = os.path.join(PROCESSED, f)
        rows = sum(1 for _ in open(path)) - 1
        size = os.path.getsize(path) / 1024
        print(f"  ✅ {f:<40} {rows:>8,} rows  ({size:.0f} KB)")


✅ Saved: data/processed/master_dataset.csv

  ALL PROCESSED FILES
  ✅ ipc_clean.csv                                 640 rows  (138 KB)
  ✅ ipc_county.csv                                 47 rows  (2 KB)
  ✅ knbs_cpi_structured.csv                        64 rows  (7 KB)
  ✅ master_dataset.csv                         13,464 rows  (2591 KB)
  ✅ nasa_monthly_clean.csv                     13,464 rows  (2385 KB)
  ✅ news_clean.csv                                300 rows  (84 KB)


---
## 6. Cleaning Summary

| Dataset | Raw Shape | Clean Shape | Key Changes |
|---|---|---|---|
| NASA POWER | 409,811 × 12 | Monthly aggregates | Parsed dates, aggregated daily→monthly, added SPI-3, season, drought features |
| FEWS NET IPC | 640 × 16 | 47 × 6 | Renamed columns, dropped ADMIN3, aggregated to county mode |
| KNBS CPI | 37 × 2 (text) | Structured table | Regex-extracted CPI + commodity prices from PDF text |
| News Articles | 300 × 5 | 300 × 12 | Parsed dates, extracted county/commodity mentions, added text features |
| **Master Dataset** | — | **Merged** | All joined on county + year + month |